# Artificial Intelligence — Lab 9
## Minimax and Adversarial Game Playing

**Course Learning Outcome — CLO5**  
Evaluate adversarial search strategies and algorithms.

**Environment:** Python 3 / Jupyter Notebook  
**Submission:** completed notebook containing predictions, traces, code, justifications, experiments, debugging answers, and reflection.

> **Assessment principle:** Correct code is only one part of the evidence. Most marks come from your ability to **reason about MAX and MIN decisions, predict utility propagation, justify recursive choices, trace a game tree, and explain why minimax produces an optimal move under its assumptions**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Game formulation | 15 min | Identify states, players, actions, terminal states, and utility |
| 2. Manual minimax trace | 25 min | Propagate utilities through a small game tree |
| 3. Implement minimax | 35 min | Build recursive MAX/MIN search |
| 4. Play against the AI | 20 min | Test decisions in a take-away game |
| 5. Analyze performance | 15 min | Count explored nodes and compare positions |
| 6. Debugging, variation & reflection | 10 min | Diagnose errors and defend conclusions |

> **Main idea:** In adversarial search, a rational agent chooses its move while assuming that the opponent will also choose the best move available to them.

## Learning Objectives

By the end of this lab, you should be able to:

1. represent a deterministic two-player zero-sum game;
2. distinguish **MAX** and **MIN** nodes;
3. define legal actions, terminal states, and utility;
4. manually propagate utilities through a minimax tree;
5. implement recursive minimax search;
6. explain why MAX chooses the maximum child value and MIN chooses the minimum;
7. identify the best move from a game state;
8. count explored game-tree nodes;
9. diagnose common minimax implementation errors;
10. justify the assumptions under which minimax is optimal.

In [ ]:
from functools import lru_cache
from typing import List, Tuple, Optional

print("Lab 9 environment ready.")

# Part I — Adversarial Search Model

A deterministic two-player game can be described using:

- **State** $s$
- **Player to move**: MAX or MIN
- **Actions** $A(s)$
- **Result function** $\operatorname{Result}(s,a)$
- **Terminal test**
- **Utility function** $U(s)$ for terminal states

The minimax value is

$$
\operatorname{Minimax}(s)=
\begin{cases}
U(s), & \text{if } s \text{ is terminal}\\
\max\limits_{a\in A(s)} \operatorname{Minimax}(\operatorname{Result}(s,a)),
& \text{if MAX moves}\\
\min\limits_{a\in A(s)} \operatorname{Minimax}(\operatorname{Result}(s,a)),
& \text{if MIN moves}
\end{cases}
$$

## Task 1.1 — MAX and MIN Reasoning

Answer in your own words.

1. Why does MAX choose the largest child value?
2. Why does MIN choose the smallest child value?
3. Why is the utility normally written from **MAX's perspective**?
4. If a terminal state has utility `+1`, what does that mean?
5. If it has utility `-1`, what does that mean?

**Your answers:**

# Part II — Manual Minimax Tree

Consider the following depth-2 game tree.

```text
                    S  (MAX)
                 /           \
              A (MIN)       B (MIN)
             /      \       /      \
           A1       A2     B1       B2
           +3       +5     +2       +9
```

The leaves are terminal utilities from MAX's perspective.

## Task 2.1 — Propagate the Utilities

Complete:

### At node A

$$
V(A)=\min(3,5)=\underline{\hspace{1.5cm}}
$$

### At node B

$$
V(B)=\min(2,9)=\underline{\hspace{1.5cm}}
$$

### At node S

$$
V(S)=\max(V(A),V(B))=\underline{\hspace{1.5cm}}
$$

Then answer:

1. Which move should MAX choose from `S`?
2. Why does MAX not choose the branch containing leaf value `9`?
3. What assumption about MIN explains this decision?

**Your answers:**

## Task 2.2 — Predict an Opponent's Response

Suppose MAX chooses branch `A`.

MIN can choose between utilities `3` and `5`.

1. Which one will MIN select?
2. Why?
3. Is MIN trying to maximize or minimize **MAX's utility**?

**Your answer:**

# Part III — A Take-Away Game

We now implement a simple game.

### Rules

- A pile contains some number of stones.
- Players alternate turns.
- On each turn, a player removes **1 or 2 stones**.
- The player who removes the **last stone wins**.

We will treat:

- AI = MAX
- Opponent = MIN

A state is:

```python
(stones_remaining, player)
```

where `player` is `"MAX"` or `"MIN"`.

## Task 3.1 — Formulate the Game

For the take-away game, identify:

- **State representation:**  
- **Initial state:**  
- **Legal actions:**  
- **Result function:**  
- **Terminal condition:**  
- **Utility function:**  

Then answer:

> Why is the player-to-move part of the state important?

**Your answer:**

In [ ]:
def actions(stones: int) -> List[int]:
    """Legal numbers of stones that can be removed."""
    return [take for take in (1, 2) if take <= stones]


def result(stones: int, player: str, take: int) -> Tuple[int, str]:
    if take not in actions(stones):
        raise ValueError("Illegal move")

    next_player = "MIN" if player == "MAX" else "MAX"
    return stones - take, next_player


def terminal_test(stones: int) -> bool:
    return stones == 0

## Task 3.2 — Define Terminal Utility

If `stones == 0`, the player whose turn it is **has already lost**, because the previous player removed the final stone.

Therefore:

- if the terminal state says `player == "MIN"`, MAX just made the winning move, so utility is `+1`;
- if the terminal state says `player == "MAX"`, MIN just made the winning move, so utility is `-1`.

Complete the function.

In [ ]:
def utility(stones: int, player: str) -> int:
    # TODO:
    # Return +1 if MAX has won,
    # -1 if MAX has lost.
    #
    # This function should only be used for terminal states.
    raise NotImplementedError

### Utility self-check

Predict both outputs before running.

In [ ]:
print("Terminal (0, MIN):", utility(0, "MIN"))
print("Terminal (0, MAX):", utility(0, "MAX"))

assert utility(0, "MIN") == 1
assert utility(0, "MAX") == -1

print("Utility tests passed.")

# Part IV — Manual Minimax Trace on the Take-Away Game

Start from:

```text
(3 stones, MAX to move)
```

MAX may remove 1 or 2 stones.

Complete the game reasoning manually.

### Option 1 — MAX removes 1

Remaining state:

```text
(2 stones, MIN)
```

What will MIN do under optimal play?

### Option 2 — MAX removes 2

Remaining state:

```text
(1 stone, MIN)
```

What will MIN do?

Then answer:

1. Which opening move should MAX choose from 3 stones?
2. What is the minimax value of `(3, MAX)`?
3. Why?

**Your answer:**

# Part V — Implement Minimax

We will first write a recursive function that returns only the minimax value.

Use the following rules:

- terminal state $\rightarrow$ utility;
- MAX node $\rightarrow$ maximum child value;
- MIN node $\rightarrow$ minimum child value.

In [ ]:
node_counter = 0

def minimax_value(stones: int, player: str) -> int:
    global node_counter
    node_counter += 1

    # TODO 1:
    # If terminal, return utility(stones, player)

    # TODO 2:
    # If player == "MAX":
    #     evaluate every legal action
    #     return the maximum child value

    # TODO 3:
    # If player == "MIN":
    #     evaluate every legal action
    #     return the minimum child value

    raise NotImplementedError

## Task 5.1 — Predict Before Testing

Predict the minimax value of:

| State | Predicted value |
|---|---:|
| `(1, MAX)` |  |
| `(1, MIN)` |  |
| `(2, MAX)` |  |
| `(3, MAX)` |  |
| `(4, MAX)` |  |

Then run the tests.

In [ ]:
tests = [
    (1, "MAX"),
    (1, "MIN"),
    (2, "MAX"),
    (3, "MAX"),
    (4, "MAX"),
]

for state in tests:
    node_counter = 0
    value = minimax_value(*state)
    print(state, "-> value:", value, "| nodes:", node_counter)

## Task 5.2 — Explain the Pattern

After running the values for 1–10 stones:

1. Which pile sizes are winning positions for MAX when MAX moves first?
2. Which are losing positions?
3. Can you identify a numerical pattern?
4. Why does the pattern emerge from allowing moves of 1 or 2 stones?

**Your answer:**

In [ ]:
for stones in range(1, 11):
    node_counter = 0
    value = minimax_value(stones, "MAX")
    print(
        f"{stones:>2} stone(s): "
        f"value={value:+d}, nodes={node_counter}"
    )

# Part VI — Return the Best Move

A game-playing agent needs an action, not only a state value.

For MAX:

$$
a^*
=
\arg\max_{a\in A(s)}
\operatorname{Minimax}(\operatorname{Result}(s,a)).
$$

Complete the function below.

In [ ]:
def best_move(stones: int) -> Tuple[int, int]:
    """Return (best_action, minimax_value_after_choice) for MAX."""

    best_action = None
    best_value = float("-inf")

    # TODO:
    # For every legal action:
    #   compute the resulting MIN state
    #   evaluate it with minimax
    #   keep the action with the greatest value

    return best_action, best_value

## Task 6.1 — Predict the Best Moves

Before running:

| Stones | Predicted best move | Why? |
|---:|---:|---|
| 3 |  |  |
| 4 |  |  |
| 5 |  |  |
| 6 |  |  |
| 7 |  |  |

Then execute.

In [ ]:
for stones in range(3, 8):
    move, value = best_move(stones)
    print(
        f"stones={stones}: "
        f"best move={move}, resulting minimax value={value}"
    )

## Task 6.2 — Explain Optimal Play

1. For which starting pile sizes can MAX force a win?
2. What does it mean to say MAX can **force** a win?
3. Is the result based on the opponent making mistakes?
4. Why does minimax assume the opponent chooses the best response?

**Your answers:**

# Part VII — Play Against the Minimax Agent

The next function lets a human play as MIN after MAX makes the first move.

You may run several games.

> If MAX starts from a winning position and minimax is correct, MAX should not lose against any legal human response.

In [ ]:
def play_game(start_stones: int):
    stones = start_stones
    player = "MAX"

    print(f"Starting with {stones} stones.")

    while stones > 0:
        print(f"\nStones remaining: {stones}")

        if player == "MAX":
            move, _ = best_move(stones)
            print("MAX removes:", move)
        else:
            legal = actions(stones)
            while True:
                try:
                    move = int(input(f"Your move {legal}: "))
                    if move in legal:
                        break
                except ValueError:
                    pass
                print("Invalid move.")

        stones, player = result(stones, player, move)

    winner = "MAX" if player == "MIN" else "MIN"
    print("\nWinner:", winner)

## Task 7.1 — Test the Agent

Play at least two games:

- one starting from a position you believe is winning for MAX;
- one starting from a position you believe is losing for MAX under perfect play.

Record:

| Initial stones | Winner | Did the outcome match minimax theory? |
|---:|---|---|
|  |  |  |
|  |  |  |

Then explain any difference between a **theoretical losing position** and an actual game where the human opponent may make a mistake.

**Your answer:**

# Part VIII — Game-Tree Size and Repeated Subproblems

Plain minimax repeatedly evaluates the same state through different action sequences.

For example, the state `(3, MAX)` may be reachable through multiple paths in a larger game tree.

We can use memoization to avoid recomputing the same state.

In [ ]:
@lru_cache(maxsize=None)
def minimax_cached(stones: int, player: str) -> int:
    if terminal_test(stones):
        return utility(stones, player)

    child_values = []

    for take in actions(stones):
        next_stones, next_player = result(stones, player, take)
        child_values.append(
            minimax_cached(next_stones, next_player)
        )

    if player == "MAX":
        return max(child_values)
    return min(child_values)

## Task 8.1 — Compare Repeated Evaluation

Run the uncached and cached versions for a larger pile.

Before running, predict which one should avoid more repeated computation.

In [ ]:
start_stones = 18

node_counter = 0
plain_value = minimax_value(start_stones, "MAX")
plain_nodes = node_counter

minimax_cached.cache_clear()
cached_value = minimax_cached(start_stones, "MAX")
cache_info = minimax_cached.cache_info()

print("Plain minimax value:", plain_value)
print("Plain recursive calls:", plain_nodes)

print("\nCached minimax value:", cached_value)
print("Cache info:", cache_info)

## Task 8.2 — Interpret the Performance Difference

1. Did both versions return the same minimax value?
2. Why should they?
3. What does memoization change: the game definition, the minimax rule, or only computation?
4. Why are repeated states possible even in a game tree?
5. What does `cache_info().hits` represent?

**Your answers:**

# Part IX — Debugging Minimax

## Task 9.1 — MAX Uses `min`

A student writes:

```python
if player == "MAX":
    return min(child_values)
```

1. What is conceptually wrong?
2. Whose preference is MAX accidentally using?
3. What should the function return instead?

**Your answer:**

## Task 9.2 — MIN Uses `max`

A student writes:

```python
if player == "MIN":
    return max(child_values)
```

1. Why is this incorrect under the standard minimax convention?
2. What should MIN do?
3. What assumption about the opponent is being violated?

**Your answer:**

## Task 9.3 — Wrong Terminal Utility

A student writes:

```python
if stones == 0:
    return 1
```

for every terminal state.

1. Why is this incorrect?
2. What additional state information is necessary?
3. Why must terminal utility depend on who made the winning move?

**Your answer:**

## Task 9.4 — Stopping One Level Too Early

A student evaluates nonterminal states using an arbitrary utility of 0 instead of continuing the search.

1. Is this full minimax?
2. What kind of additional concept would be needed to make early stopping meaningful?
3. Why is exact terminal utility different from a heuristic evaluation function?

**Your answer:**

# Part X — Personalized Game Variation

Use the last digit of your student ID.

- `0–3`: allowed moves = `{1, 2}`
- `4–6`: allowed moves = `{1, 3}`
- `7–9`: allowed moves = `{1, 2, 3}`

For this section only, create a new action function using your assigned move set.

In [ ]:
LAST_DIGIT = None  # TODO: replace with an integer from 0 to 9

if LAST_DIGIT is not None:
    if 0 <= LAST_DIGIT <= 3:
        PERSONAL_MOVES = (1, 2)
    elif 4 <= LAST_DIGIT <= 6:
        PERSONAL_MOVES = (1, 3)
    elif 7 <= LAST_DIGIT <= 9:
        PERSONAL_MOVES = (1, 2, 3)
    else:
        raise ValueError("LAST_DIGIT must be between 0 and 9")

    print("Assigned move set:", PERSONAL_MOVES)

## Task 10.1 — Predict Before Coding

For your move set:

1. Predict which pile sizes from 1 to 8 are losing positions for MAX.
2. State the pattern you expect, if any.
3. Explain why changing the action set can change the winning/losing structure of the game.

**Your prediction:**

In [ ]:
def personal_actions(stones: int):
    if LAST_DIGIT is None:
        return []
    return [take for take in PERSONAL_MOVES if take <= stones]


@lru_cache(maxsize=None)
def personal_minimax(stones: int, player: str) -> int:
    if stones == 0:
        return 1 if player == "MIN" else -1

    child_values = []

    for take in personal_actions(stones):
        next_player = "MIN" if player == "MAX" else "MAX"
        child_values.append(
            personal_minimax(stones - take, next_player)
        )

    if player == "MAX":
        return max(child_values)
    return min(child_values)

In [ ]:
if LAST_DIGIT is not None:
    personal_minimax.cache_clear()

    for stones in range(1, 9):
        value = personal_minimax(stones, "MAX")
        print(
            f"{stones:>2} stone(s): "
            f"{'WIN' if value == 1 else 'LOSS'}"
        )

## Task 10.2 — Explain the Personalized Result

1. Was your predicted pattern correct?
2. Which pile sizes are losing positions?
3. Why did changing the action set change the game tree?
4. Which part of the game formulation changed?
5. Did the minimax principle itself change?

**Your answers:**

# Part XI — Individual Understanding Check

Your instructor may ask one short question about your notebook.

Possible prompts:

- Show me where MAX chooses the maximum value.
- Show me where MIN chooses the minimum value.
- Why does terminal utility depend on the player to move?
- Explain what it means for MAX to force a win.
- Why does minimax assume an optimal opponent?
- What does memoization change?
- Why can the same game state be reached through multiple paths?
- Which part of the game changed in your personalized variation?

> You are expected to explain the **AI concept represented by the code**, not memorize Python syntax.

# Reflection

Answer concisely but precisely.

### R1 — Optimal Opponent
Why does minimax plan against the opponent's best possible response rather than an average response?

**Answer:**

### R2 — Utility
Why are terminal utilities sufficient for full minimax in a small finite game?

**Answer:**

### R3 — Game Tree
Why can minimax become computationally expensive as game depth and branching factor grow?

**Answer:**

### R4 — Memoization
Why can caching improve performance without changing the decision returned by minimax?

**Answer:**

### R5 — Limitations
Name one assumption of this lab's minimax model that does not hold in many real games.

**Answer:**

# Submission Checklist

Before submitting, verify that your notebook contains:

- [ ] MAX/MIN reasoning;
- [ ] manual utility propagation on the small game tree;
- [ ] take-away game formulation;
- [ ] working terminal utility;
- [ ] manual minimax trace for 3 stones;
- [ ] working `minimax_value`;
- [ ] winning/losing pattern analysis;
- [ ] working `best_move`;
- [ ] at least two game tests;
- [ ] memoization comparison;
- [ ] debugging answers;
- [ ] personalized move-set experiment;
- [ ] prediction before personalized execution;
- [ ] reflection answers;
- [ ] visible outputs from important code cells.

Suggested filename:

```text
Lab09_StudentID.ipynb
```

# Assessment Guide — 10 Marks

| Component | Marks | Evidence expected |
|---|---:|---|
| **Correct implementation** | **2.0** | Minimax value function and best-move selection work correctly |
| **Algorithmic justification** | **3.0** | Explains MAX/MIN choices, utility, recursion, optimal response, and game formulation |
| **Experimental analysis** | **2.0** | Interprets winning positions, game outcomes, and memoization results |
| **Trace / prediction / debugging** | **1.0** | Manual minimax propagation, predictions, and faulty-code diagnosis |
| **Individual understanding check** | **1.0** | Short explanation of selected part of the student's own work |
| **Code quality & completeness** | **1.0** | Readable code, complete responses, required outputs |
| **Total** | **10.0** |  |

> **Key rule:** Correct code without adequate explanation earns only a limited portion of the marks.

## Key Takeaways

- Minimax models rational play in deterministic adversarial games.
- MAX chooses the child with the largest utility.
- MIN chooses the child with the smallest utility from MAX's perspective.
- Terminal utilities are propagated backward through the game tree.
- Under its assumptions, minimax chooses an optimal move against an optimal opponent.
- Game trees can grow quickly because of branching and depth.
- Memoization can reduce repeated computation without changing the decision.
- Changing the legal action set changes the game tree, but not the minimax principle.

The next lab will extend this work with **Alpha-Beta Pruning and Evaluation Functions**.